# 07 — Khulna Paper-Style ML Downscaling (FINAL FIXED)

Corrected end-to-end modelling stage.

**Methodological safeguards**
- Training/validation/test station table comes from native raster extraction (Notebook 06).
- `Comb1` uses exactly 8 precipitation products.
- `Comb2` uses CHIRPS + PERSIANN-CDR (`CDR`) + ERA5.
- Standalone `PERSIANN` is not used.
- Adapted land set: DEM + NDVI + LST Day + Distance to Sea.
- 2017–2020 train, 2021 validation, 2022 independent test.
- Hyperparameters are selected on validation only.
- 2022 spatial rasters come from Notebook 05 target-grid outputs.
- No edge clamping and no nearest fill of NoData.
- Existing output TIFFs are overwritten by default.
- `0.005°` is reported as **grid spacing**, not guaranteed effective rainfall information resolution.

In [ ]:
from pathlib import Path
import warnings

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for d in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
import json, math, warnings, joblib
import numpy as np
import pandas as pd
import rasterio
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid

TRAIN_YEARS=[2017,2018,2019,2020]
VALID_YEARS=[2021]
TEST_YEARS=[2022]

PRECIP8 = ["CCS","PDIR","GSMaP_MVK","CDR","CHIRPS","IMERG","GSMaP_Gauge_v7","ERA5"]
LAND = ["DEM","NDVI","LST_Day","Distance_Sea"]

COMBINATIONS = {
    "Comb1": PRECIP8,
    "Comb1_land": PRECIP8 + LAND,
    "Comb2": ["CHIRPS","CDR","ERA5"],
    "Comb2_land": ["CHIRPS","CDR","ERA5"] + LAND,
}

SAMPLES_PATH = PROCESSED_DIR/"station_samples_native_tidy.csv"
GRID_ROOT = PROCESSED_DIR/"test2022_target_grid"
RUN_DIR = OUTPUT_DIR/"paper_style_FIXED"
RUN_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUT = MODEL_DIR/"paper_style_FIXED"
MODEL_OUT.mkdir(parents=True, exist_ok=True)

OVERWRITE = True
RANDOM_STATE = 42

if not SAMPLES_PATH.exists():
    raise FileNotFoundError("Run Notebook 06 first.")
if not GRID_ROOT.exists():
    raise FileNotFoundError("Run Notebook 05 first.")

data = pd.read_csv(SAMPLES_PATH)
print("Samples:", data.shape)

In [ ]:
# Optional model libraries.
model_builders = {
    "RandomForest": lambda p: RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **p)
}
param_grids = {
    "RandomForest": {
        "n_estimators":[300,600],
        "max_depth":[None,10,20],
        "min_samples_leaf":[1,2,4],
        "max_features":["sqrt",0.8],
    }
}

try:
    from xgboost import XGBRegressor
    model_builders["XGBoost"] = lambda p: XGBRegressor(
        random_state=RANDOM_STATE, n_jobs=-1, objective="reg:squarederror", **p
    )
    param_grids["XGBoost"] = {
        "n_estimators":[300,600], "max_depth":[3,6],
        "learning_rate":[0.03,0.08], "subsample":[0.8,1.0],
        "colsample_bytree":[0.8,1.0]
    }
except Exception as e:
    print("XGBoost unavailable:", e)

try:
    from lightgbm import LGBMRegressor
    model_builders["LightGBM"] = lambda p: LGBMRegressor(
        random_state=RANDOM_STATE, verbosity=-1, **p
    )
    param_grids["LightGBM"] = {
        "n_estimators":[300,600], "learning_rate":[0.03,0.08],
        "num_leaves":[15,31], "max_depth":[-1,10]
    }
except Exception as e:
    print("LightGBM unavailable:", e)

try:
    from catboost import CatBoostRegressor
    model_builders["CatBoost"] = lambda p: CatBoostRegressor(
        random_seed=RANDOM_STATE, verbose=False, **p
    )
    param_grids["CatBoost"] = {
        "iterations":[300,600], "depth":[4,6,8], "learning_rate":[0.03,0.08]
    }
except Exception as e:
    print("CatBoost unavailable:", e)

print("Models:", list(model_builders))

In [ ]:
def pcc(y, pred):
    y=np.asarray(y,float); pred=np.asarray(pred,float)
    if len(y)<2 or np.std(y)==0 or np.std(pred)==0:
        return np.nan
    return float(np.corrcoef(y,pred)[0,1])

def metrics(y,pred):
    return {
        "MAE":mean_absolute_error(y,pred),
        "RMSE":mean_squared_error(y,pred)**0.5,
        "PCC":pcc(y,pred),
        "R2":r2_score(y,pred),
    }

def get_split(df, years, features):
    d=df[df.year.isin(years)].dropna(subset=["rainfall_mm"]+features).copy()
    return d, d[features].to_numpy(float), d["rainfall_mm"].to_numpy(float)

results=[]
best_models={}

for combo, features in COMBINATIONS.items():
    print("\n===",combo,"===")
    tr,Xtr,ytr = get_split(data,TRAIN_YEARS,features)
    va,Xva,yva = get_split(data,VALID_YEARS,features)
    te,Xte,yte = get_split(data,TEST_YEARS,features)

    print("rows train/val/test:",len(tr),len(va),len(te))
    if min(len(tr),len(va),len(te)) < 10:
        raise ValueError(f"{combo}: too few complete rows after native extraction.")

    for model_name,builder in model_builders.items():
        best=None
        for params in ParameterGrid(param_grids[model_name]):
            model=builder(params)
            model.fit(Xtr,ytr)
            pv=np.clip(model.predict(Xva),0,None)
            score=mean_squared_error(yva,pv)**0.5
            if best is None or score < best["val_rmse"]:
                best={"val_rmse":score,"params":params,"model":model}

        # Test once with model selected from validation.
        pv=np.clip(best["model"].predict(Xva),0,None)
        pt=np.clip(best["model"].predict(Xte),0,None)
        vm=metrics(yva,pv); tm=metrics(yte,pt)

        row={"combination":combo,"model":model_name,
             "n_train":len(tr),"n_valid":len(va),"n_test":len(te),
             **{f"valid_{k}":v for k,v in vm.items()},
             **{f"test_{k}":v for k,v in tm.items()},
             "best_params":json.dumps(best["params"])}
        results.append(row)

        # Refit the selected configuration on train + validation for 2022 spatial prediction.
        tv=pd.concat([tr,va],ignore_index=True)
        final_model=builder(best["params"])
        final_model.fit(tv[features].to_numpy(float),tv["rainfall_mm"].to_numpy(float))
        best_models[(combo,model_name)] = final_model
        joblib.dump(final_model, MODEL_OUT/f"{combo}__{model_name}.joblib")

results_df=pd.DataFrame(results).sort_values(["combination","valid_RMSE"])
display(results_df)
results_df.to_csv(RUN_DIR/"model_metrics.csv",index=False)

In [ ]:
# Select one model per combination by validation RMSE only.
selected = (
    results_df.sort_values(["combination","valid_RMSE"])
              .groupby("combination",as_index=False)
              .first()
)
display(selected)
selected.to_csv(RUN_DIR/"selected_models_by_validation.csv",index=False)

In [ ]:
# Test-year station predictions for the selected models.
test_pred_rows=[]
for _,s in selected.iterrows():
    combo=s["combination"]; model_name=s["model"]
    features=COMBINATIONS[combo]
    te,Xte,yte=get_split(data,TEST_YEARS,features)
    model=best_models[(combo,model_name)]
    pred=np.clip(model.predict(Xte),0,None)
    tmp=te[["station_id","year","month","rainfall_mm","latitude","longitude"]].copy()
    tmp["combination"]=combo
    tmp["model"]=model_name
    tmp["predicted_mm"]=pred
    tmp["error_mm"]=pred-tmp["rainfall_mm"]
    test_pred_rows.append(tmp)

test_predictions=pd.concat(test_pred_rows,ignore_index=True)
test_predictions.to_csv(RUN_DIR/"test2022_station_predictions.csv",index=False)
display(test_predictions.head())

In [ ]:
# Helpers for target-grid feature loading.
def grid_path(feature, month):
    if feature in ["DEM","Distance_Sea"]:
        return GRID_ROOT/"static"/f"{feature}.tif"
    return GRID_ROOT/feature/f"{feature}_2022_{month:02d}.tif"

def read_grid(path):
    with rasterio.open(path) as src:
        a=src.read(1).astype("float32")
        nod=src.nodata
        valid=np.isfinite(a)
        if nod is not None:
            valid &= ~np.isclose(a,nod)
        return a,valid,src.profile.copy()

def predict_month(model,features,month,out_path):
    arrays=[]; masks=[]; profile=None
    for f in features:
        p=grid_path(f,month)
        if not p.exists():
            raise FileNotFoundError(p)
        a,v,pr=read_grid(p)
        if profile is None:
            profile=pr
            shape=a.shape
        elif a.shape != shape:
            raise ValueError(f"Grid mismatch: {p}")
        arrays.append(a); masks.append(v)

    valid=np.logical_and.reduce(masks)
    out=np.full(shape,-9999.0,dtype="float32")
    if valid.any():
        X=np.column_stack([a[valid] for a in arrays])
        good=np.all(np.isfinite(X),axis=1)
        pred=np.full(X.shape[0],np.nan,dtype="float32")
        pred[good]=np.clip(model.predict(X[good]),0,None).astype("float32")
        vals=np.full(X.shape[0],-9999.0,dtype="float32")
        vals[np.isfinite(pred)]=pred[np.isfinite(pred)]
        out[valid]=vals

    profile.update(dtype="float32",nodata=-9999.0,count=1,compress="deflate",predictor=2)
    out_path.parent.mkdir(parents=True,exist_ok=True)
    with rasterio.open(out_path,"w",**profile) as dst:
        dst.write(out,1)
    coverage=100*np.sum(out!=-9999.0)/np.sum(valid | ~valid)  # overall target raster
    return coverage

In [ ]:
# Spatial monthly + annual maps. Existing files are overwritten when OVERWRITE=True.
spatial_qc=[]

for _,s in selected.iterrows():
    combo=s["combination"]; model_name=s["model"]
    model=best_models[(combo,model_name)]
    features=COMBINATIONS[combo]
    combo_dir=RUN_DIR/"maps"/combo
    monthly_paths=[]

    for month in range(1,13):
        out=combo_dir/f"downscaled_2022_{month:02d}.tif"
        if out.exists() and not OVERWRITE:
            monthly_paths.append(out)
            continue
        coverage=predict_month(model,features,month,out)
        spatial_qc.append({"combination":combo,"month":month,"coverage_pct_all_grid":coverage})
        monthly_paths.append(out)

    # Annual sum only where ALL 12 months are valid.
    stack=[]; valid_stack=[]; profile=None
    for p in monthly_paths:
        a,v,pr=read_grid(p)
        stack.append(a); valid_stack.append(v)
        profile=pr
    all_valid=np.logical_and.reduce(valid_stack)
    annual=np.full(stack[0].shape,-9999.0,dtype="float32")
    annual[all_valid]=np.sum([a[all_valid] for a in stack],axis=0).astype("float32")

    annual_path=combo_dir/"annual_downscaled_2022.tif"
    profile.update(dtype="float32",nodata=-9999.0,count=1,compress="deflate",predictor=2)
    with rasterio.open(annual_path,"w",**profile) as dst:
        dst.write(annual,1)

pd.DataFrame(spatial_qc).to_csv(RUN_DIR/"spatial_prediction_qc.csv",index=False)
print("Spatial outputs:", RUN_DIR/"maps")

In [ ]:
# Basic annual physical/QC summary to make artifacts obvious before GIS layout.
summary=[]
for combo in COMBINATIONS:
    p=RUN_DIR/"maps"/combo/"annual_downscaled_2022.tif"
    with rasterio.open(p) as src:
        a=src.read(1,masked=True)
        v=a.compressed()
        summary.append({
            "combination":combo,
            "valid_pixels":v.size,
            "min_mm":float(v.min()) if v.size else np.nan,
            "p05_mm":float(np.percentile(v,5)) if v.size else np.nan,
            "median_mm":float(np.median(v)) if v.size else np.nan,
            "p95_mm":float(np.percentile(v,95)) if v.size else np.nan,
            "max_mm":float(v.max()) if v.size else np.nan,
            "mean_mm":float(v.mean()) if v.size else np.nan,
        })

annual_summary=pd.DataFrame(summary)
display(annual_summary)
annual_summary.to_csv(RUN_DIR/"annual_2022_summary.csv",index=False)

print("""
FINAL NOTE:
0.005 degree is the output grid spacing. The effective spatial information content
is limited by the native resolutions of the precipitation and predictor datasets.
Inspect monthly maps and alignment_qc.csv before interpreting fine-scale gradients.
""")